In [0]:
# 19_daily_update_news_and_market

import sys
import importlib
import pandas as pd
import random
import numpy as np

SRC_PATH = "/Workspace/Users/ariamostajeran99@gmail.com/stock_project_V2/stock-mlops-databricks/src"
if SRC_PATH not in sys.path:
    sys.path.append(SRC_PATH)

import config
import universe
import news_updater
# import your market loader module here if you already have one
# import data_loader

importlib.reload(config)
importlib.reload(universe)
importlib.reload(news_updater)
# importlib.reload(data_loader)

from config import RAW_NEWS_TABLE_NAME, RAW_TABLE_NAME
from universe import TRAIN_UNIVERSE, ALL_SYMBOLS
from news_updater import NewsUpdater
# from data_loader import MarketDataLoader

# optional reproducibility
GLOBAL_RANDOM_SEED = getattr(config, "GLOBAL_RANDOM_SEED", 42)
random.seed(GLOBAL_RANDOM_SEED)
np.random.seed(GLOBAL_RANDOM_SEED)

In [0]:
FINNHUB_API_KEY = "d6tbsj9r01qhkb43ho60d6tbsj9r01qhkb43ho6g"

# one-time backfill mode:
RUN_FULL_NEWS_BACKFILL = False  # set to True only once
BACKFILL_DAYS = 365

# daily mode:
DAILY_LOOKBACK_DAYS = 3

In [0]:
# ===== UPDATE NEWS =====

updater = NewsUpdater(
    api_key=FINNHUB_API_KEY,
    train_universe=TRAIN_UNIVERSE
)

if RUN_FULL_NEWS_BACKFILL:
    print(f"Running one-time Finnhub backfill for last {BACKFILL_DAYS} days...")
    new_news_df = updater.fetch_last_n_days(n_days=BACKFILL_DAYS)
else:
    print(f"Running daily Finnhub update for last {DAILY_LOOKBACK_DAYS} days...")
    new_news_df = updater.fetch_last_n_days(n_days=DAILY_LOOKBACK_DAYS)

print("Fetched new news rows:", len(new_news_df))
print(new_news_df.head(5))

In [0]:
# ===== LOAD EXISTING NEWS TABLE IF IT EXISTS =====

table_exists = False
tables_df = spark.sql("SHOW TABLES").toPandas()

if RAW_NEWS_TABLE_NAME in tables_df["tableName"].tolist():
    table_exists = True

if table_exists:
    existing_news_df = spark.table(RAW_NEWS_TABLE_NAME).toPandas()
    print("Existing news rows:", len(existing_news_df))
else:
    existing_news_df = pd.DataFrame()
    print("News table does not exist yet. Creating from scratch.")

In [0]:
# ===== APPEND + SAVE UPDATED NEWS =====

updated_news_df = updater.append_to_existing_news(existing_news_df, new_news_df)

spark.createDataFrame(updated_news_df) \
    .write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(RAW_NEWS_TABLE_NAME)

print("Saved updated news table.")
print("Updated news rows:", len(updated_news_df))
print(updated_news_df["Ticker"].value_counts())
print(updated_news_df["Date"].min(), updated_news_df["Date"].max())